# Imports

In [80]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# API
import requests
import json

# Progress bar
from tqdm import tqdm

# Paths
import os
data_path = os.path.join('..', 'data')

# Query gene symbols
import mygene

# McNemar test and multiple testing correction
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar

# Gene Set Enrichment Analysis
from goatools.obo_parser import GODag
import gseapy as gp
from gseapy import gsea

# Load Data

In [146]:
test_df = pd.read_pickle(os.path.join(data_path, '2_test_nn_with_baseline_50.pkl'))
test_df = test_df.head(15).copy()

In [147]:
test_df.head()

,cluster,length (aa),query_gene,query_transcript,seq,amino_acid_seq,median,subject_transcript,average_pident,gene,transcript,median_float,mfc_prediction,mfc_accuracy,mfb_prediction,mfb_accuracy,exp_harmonized_prediction,exp_harmonized_accuracy
0,2191,932,ENSG00000134324,ENST00000396097,ATGAGCAGAGTGCAGACCATGAATTACGTGGGGCAGTTAGCCGGCC...,MSRVQTMNYVGQLAGQVFVTVKELYKGLNPATLSGCIDIIVIRQPN...,"(expr_pre75_90, expr_pre50_75, expr_pre75_90, ...",ENST00000261596,48.943442,ENSG00000134324,ENST00000396097,"(2.880000114440918, 1.5, 2.2899999618530273, 0...",ATGAGCAGAGTGCAGACCATGAACTACGTGGGCCAGCTGGCCGGCC...,0.457663,ATGAGCAGGGTGCAGACCATGAATTATGTGGGCCAGCTGGCCGGCC...,0.456592,ATGAGCAGAGTGCAGACCATGAACTACGTGGGCCAGCTGGCCGGCC...,0.457663
1,5329,249,ENSG00000172456,ENST00000634606,ATGTGGCTGGACCATCGAGCAGTCAGTCAAGTTAACAGGATCAATG...,MWLDHRAVSQVNRINETKHSVLQYVGGVMSVEMQAPKLLWLKENLR...,"(expr_pre25_50, expr_pre25_50, expr_pre25_50, ...",NaN,0.000000,ENSG00000172456,ENST00000634606,"(0.029999999329447746, 0.019999999552965164, 0...",ATGTGGCTGGACCACAGAGCCGTGAGCCAGGTGAACAGAATCAACG...,0.360000,ATGTGGCTGGACCACAGGGCCGTGTCCCAGGTGAACAGAATTAATG...,0.348000,ATGTGGCTGGACCACAGAGCCGTGAGCCAGGTGAACAGAATCAACG...,0.360000
2,7096,463,ENSG00000118495,ENST00000416623,ATGGCCACGTTCCCCTGCCAGTTATGTGGCAAGACGTTCCTCACCC...,MATFPCQLCGKTFLTLEKFTIHNYSHSRERPYKCVQPDCGKAFVSR...,"(expr_pre25_50, expr_pre50_75, expr_pre50_75, ...",ENST00000367405,18.618728,ENSG00000118495,ENST00000416623,"(0.1599999964237213, 0.27000001072883606, 0.86...",ATGGCCACCTTCCCCTGCCAGCTGTGCGGCAAGACCTTCCTGACCC...,0.521552,ATGGCCACCTTCCCCTGCCAGCTGTGCGGCAAGACCTTCCTGACCC...,0.521552,ATGGCCACCTTCCCCTGCCAGCTGTGCGGCAAGACCTTCCTGACCC...,0.521552
3,7405,432,ENSG00000257950,ENST00000550383,ATGGGGCAGGCGGGCTGCAAGGGGCTCTGCCTGTCGCTGTTCGACT...,MGQAGCKGLCLSLFDYKTEKYVIAKNKKVGLLYRLLQASILAYLVV...,NaN,ENST00000413302,38.319426,ENSG00000257950,ENST00000550383,NaN,ATGGGCCAGGCCGGCTGCAAGGGCCTGTGCCTGAGCCTGTTCGACT...,0.598152,ATGGGCCAGGCCGGCTGCAAGGGCCTGTGCCTGAGCCTGTTCGACT...,0.581986,ATGGGCCAGGCCGGCTGCAAGGGCCTGTGCCTGAGCCTGTTCGACT...,0.598152
4,5500,57,ENSG00000107669,ENST00000690706,ATGGCTTTCTGGGCGGGGGGTTCGCCCAGCGTCGTGGACTATTTCC...,MAFWAGGSPSVVDYFPSEDFYRCGYCKNESGSRSNGMWAHSMTVQD...,NaN,NaN,0.000000,ENSG00000107669,ENST00000690706,NaN,ATGGCCTTCTGGGCCGGCGGCAGCCCCAGCGTGGTGGACTACTTCC...,0.551724,ATGGCCTTCTGGGCCGGCGGCAGCCCCAGCGTGGTGGACTACTTCC...,0.551724,ATGGCCTTCTGGGCCGGCGGCAGCCCCAGCGTGGTGGACTACTTCC...,0.551724


# Calculate Genes Scores

## Calculate McNemar P-value per Transcript

### Get (Prediction == Correct) Bit Vectors For Models

In [148]:
def split_to_codons(nt_sequence):
    return [nt_sequence[i:i+3] for i in range(0, len(nt_sequence), 3)]

def compare_codons(ground_truth_codons_column, predicted_codons_column):
    ground_truth_codons_column = ground_truth_codons_column.apply(lambda x: np.array(x))
    predicted_codons_column = predicted_codons_column.apply(lambda x: np.array(x))
    comparison_results = ground_truth_codons_column.combine(predicted_codons_column, lambda x, y: np.where(x == y, 1, 0))
    return comparison_results

In [149]:
mfc_codons = test_df['mfc_prediction'].apply(split_to_codons)
mfb_codons = test_df['mfb_prediction'].apply(split_to_codons)
ground_truth_codons = test_df['seq'].apply(split_to_codons)

# Correct/incorrect per codon for two models (1=correct, 0=incorrect)
test_df['is_mfc_correct_bit_vec'] = compare_codons(ground_truth_codons, mfc_codons)
test_df['is_mfb_correct_bit_vec'] = compare_codons(ground_truth_codons, mfb_codons)

print("Is MFC correct bit vectors (head):")
print(test_df['is_mfc_correct_bit_vec'].head())

Is MFC correct bit vectors (head):
0    [1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, ...
1    [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...
2    [1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, ...
3    [1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, ...
4    [1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, ...
Name: is_mfc_correct_bit_vec, dtype: object


### Calculate Adjusted Median McNemar P values

In [150]:
# calculate McNemar's P-value
def calculate_mcnemar_p_value(model1_bit_vec, model2_bit_vec):
    # Contingency table
    n_10 = sum((m1 == 1 and m2 == 0) for m1, m2 in zip(model1_bit_vec, model2_bit_vec))
    n_01 = sum((m1 == 0 and m2 == 1) for m1, m2 in zip(model1_bit_vec, model2_bit_vec))
    # McNemar's test only cares about off-diagonal elements
    table = [[0, n_10], [n_01, 0]]

    # McNemar's test
    result = mcnemar(table, exact=True)
    p_value = result.pvalue

    return p_value

# Compute p-value for pandas row
def calculate_p_value_row(row):
    p_value = calculate_mcnemar_p_value(row['is_mfc_correct_bit_vec'], row['is_mfb_correct_bit_vec'])
    return p_value

# Calculate adjusted median p-values for each gene symbol and score each gene
def calculate_gene_scores(test_df):
    test_df['p_value'] = test_df.apply(calculate_p_value_row, axis=1)
    
    # Pick median p_value for each gene symbol
    median_p_values = test_df.groupby('query_gene_symbol')['p_value'].median()
    test_df = test_df.drop_duplicates(subset='query_gene_symbol')
    test_df = test_df.merge(median_p_values.rename('median_p_value'), on='query_gene_symbol')

    # Adjust median p-values for multiple testing
    test_df['adjusted_median_p_value'] = multipletests(test_df['median_p_value'], method='fdr_bh')[1]

    # Score each gene
    acc_diff = test_df['mfc_accuracy'] - test_df['mfb_accuracy']
    epsilon = 1e-10  # Small value to avoid log(0)
    test_df['score'] = -np.log10(test_df['adjusted_median_p_value']) * -np.log10(np.abs(acc_diff) + epsilon) * np.sign(acc_diff)

    return test_df[['query_gene_symbol', 'score']]

In [153]:
# Apply the function to each row
test_df['query_gene_symbol'] = test_df['query_gene']

scored_genes = calculate_gene_scores(test_df)

scored_genes = scored_genes.sort_values(by='score', ascending=True)

# Replace Gene IDs with Gene Symbols

In [ ]:
# Initialize MyGeneInfo
mg = mygene.MyGeneInfo()

# List of Ensembl Gene IDs
ensembl_gene_ids = test_df['query_gene'].drop_duplicates()

# Query MyGeneInfo for gene symbols
result = mg.querymany(ensembl_gene_ids, scopes="ensembl.gene", fields="symbol", species="human")

# Create a dictionary mapping Ensembl IDs to gene symbols
ensembl_to_symbol = {entry["query"]: entry.get("symbol", "N/A") for entry in result}

# Map Ensembl IDs to gene symbols
scored_genes['query_gene_symbol'] = scored_genes['query_gene_symbol'].map(ensembl_to_symbol)

# Drop entries with no match
scored_genes = scored_genes[scored_genes['query_gene_symbol'] != "N/A"]

In [157]:
scored_genes

,query_gene_symbol,score
11,HECTD1,-0.189412
12,MS4A6A,-0.162238
5,CFTR,-0.110229
6,NSF,-0.102646
4,ATE1,-0.000000
13,SLC11A1,-0.000000
0,LPIN1,-0.000000
2,PLAGL1,-0.000000
9,MOCS2,0.000000
10,RHOG,0.085796


# Gene Set Enrichment Analysis

In [ ]:
# all_possible_namespaces = gp.get_library_name()
relevant_namespaces = ['GO_Molecular_Function_2023', 'GO_Biological_Process_2023', 'GO_Cellular_Component_2023'] #, 'WikiPathways_2024_Human', 'GTEx_Tissues_V8_2023']

pre_res = gp.prerank(rnk=scored_genes, gene_sets=relevant_namespaces[0], processes=4, permutation_num=1000, outdir='gsea_prerank', seed=42)

# Retrieve GO terms

In [10]:
def quickgo_batch_query(go_ids, aspect="molecular_function"):
    url = "https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms"
    headers = {"Accept": "application/json"}
    params = {
        "ids": ",".join(go_ids),  # Comma-separated GO IDs
        "relationType": "exact"  # Restrict to exact terms
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code == 200:
        results = response.json()["results"]
        results_df = pd.DataFrame([entry for entry in results ])
        results_df = results_df[results_df['isObsolete'] != True]
        results_df = results_df[["id", "name", "definition", "aspect"]]
        return results_df
    else:
        print("Error:", response.status_code, response.text)
        return None

## Utils

In [3]:
def get_go_namespace(go_id):
    """
    Retrieve (from ENSEMBL API) the namespace of a given Gene Ontology (GO) term:
    'molecular_function', 'biological_process', or 'cellular_component'.
    Args:
        go_id (str): The GO term ID to look up.
    Returns:
        str: The namespace of the GO term if the request is successful.
        None: If the request fails or the GO term is not found.
    """
    
    url = f"https://rest.ensembl.org/ontology/id/{go_id}?content-type=application/json"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data['namespace']  # Returns 'molecular_function', 'biological_process', or 'cellular_component'
    else:
        return None

def get_go_terms_from_ensembl(transcript_id, verbose=True):
    """
    Retrieve (from ENSEMBL API) Gene Ontology (GO) terms for a given Ensembl transcript ID.
    Parameters:
        transcript_id (str): The Ensembl transcript ID for which to retrieve GO terms.
        verbose (bool, optional): If True, prints additional information in case of failure. Default is False.
    Returns:
        list: A list of GO term IDs associated with the given transcript ID. Returns an empty list if no GO terms are found or if the request fails.
    Example:
    >>> get_go_terms_from_ensembl("ENST00000367770")
    ['GO:0003677', 'GO:0006355', 'GO:0005634']
    """

    # The Ensembl xrefs endpoint provides GO terms for a transcript
    url = f"https://rest.ensembl.org/xrefs/id/{transcript_id}?expand=1;content-type=application/json"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        go_terms = []
        for entry in data:
            if entry['dbname'] == 'GO':  # Ensure we get only GO terms
                go_terms.append(entry['primary_id'])
        return set(go_terms)  # Return a set to remove duplicates
    else:
        if verbose:
            print(f"Failed to retrieve data for {transcript_id}. Status code: {response.status_code}")
        return []


def get_molecular_function_term_from_ensembl(transcript_id, verbose=False):
    """
    Retrieve (from ENSEMBL API) the molecular function Gene Ontology (GO) term for a given Ensembl transcript ID.
    Parameters:
        transcript_id (str): The Ensembl transcript ID for which to retrieve the molecular function GO term.
        verbose (bool, optional): If True, prints additional information in case of failure. Default is False.
    Returns:
        str: The molecular function GO term ID associated with the given transcript ID. Returns None if no molecular function term is found or if the request fails.
    Example:
    >>> get_molecular_function_term_from_ensembl("ENST00000367770")
    'GO:0003677'
    """

    go_terms = get_go_terms_from_ensembl(transcript_id, verbose=verbose)
    if verbose:
        print(f"GO terms for {transcript_id}: {go_terms}")
    for go_id in go_terms:
        namespace = get_go_namespace(go_id)
        if namespace == 'molecular_function':
            return go_id
    return None

In [ ]:
get_molecular_function_term_from_ensembl("ENST00000373020", verbose=True)

{'GO:0070062', 'GO:0043123', 'GO:0039532', 'GO:0016020', 'GO:0043124', 'GO:0005515', 'GO:0005886'}


'GO:0005515'

In [5]:
test_df.head(15)['query_transcript'].apply(get_go_terms_from_ensembl)

0     {GO:0019432, GO:0005654, GO:0005634, GO:003196...
1                              {GO:0016301, GO:0005975}
2     {GO:0046872, GO:0001228, GO:0005654, GO:000012...
3                                                    {}
4                              {GO:0004057, GO:0016746}
5     {GO:0019321, GO:0005737, GO:0005975, GO:000557...
6     {GO:0034707, GO:0055038, GO:0006821, GO:000525...
7     {GO:0046872, GO:0035494, GO:0016787, GO:000552...
8     {GO:0140374, GO:0042802, GO:0051607, GO:004847...
9     {GO:0007155, GO:0015813, GO:0034755, GO:000682...
10    {GO:0005829, GO:0030366, GO:0000166, GO:000551...
11    {GO:0043652, GO:0016601, GO:0031410, GO:009879...
12     {GO:0016567, GO:0016740, GO:0006511, GO:0061630}
13    {GO:0005802, GO:0007166, GO:0016020, GO:000588...
14                                                   {}
Name: query_transcript, dtype: object

In [19]:
from goatools.obo_parser import GODag
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

# Load the GO DAG
go_dag = GODag("go-basic.obo")

# Gene-to-GO annotations
go_annotations = {
    "GeneA": {"GO:0003677", "GO:0005524"},  # Use sets for GO terms
    "GeneB": {"GO:0003677"},
}

# Background (population) genes and study genes
all_genes = set(go_annotations.keys())  # Background
study_genes = {"GeneA"}  # Genes of interest

# Perform GO enrichment analysis
goea = GOEnrichmentStudyNS(
    all_genes,  # Population genes
    {"molecular_function": go_annotations},  # Namespace to associations
    go_dag,
    propagate_counts=True,
    alpha=0.05,  # Significance cutoff
    methods=["fdr_bh"]  # Multiple test correction
)

results = goea.run_study(study_genes)

# Print significant results
for r in results:
    if r.p_fdr_bh < 0.05:  # Filter significant terms
        print(f"{r.GO}\t{r.goterm.name}\t{r.p_fdr_bh}")


go-basic.obo: fmt(1.2) rel(2024-10-27) 44,017 Terms

Load molecular_function Ontology Enrichment Analysis ...
Propagating term counts up: is_a
100%      2 of      2 population items found in association

Runing molecular_function Ontology Analysis: current study set of 1 IDs.
100%      1 of      1 study items found in association
100%      1 of      1 study items found in population(2)
Calculating 19 uncorrected p-values using fisher_scipy_stats
      19 terms are associated with      2 of      2 population items
      19 terms are associated with      1 of      1 study items
  METHOD fdr_bh:
       0 GO terms found significant (< 0.05=alpha) (  0 enriched +   0 purified): statsmodels fdr_bh
       0 study items associated with significant GO IDs (enriched)
       0 study items associated with significant GO IDs (purified)


In [ ]:
# Load the GO DAG
go_dag = GODag("go-basic.obo")

# Assume the dataframe is named `test_df` and contains the required columns
# test_df = pd.DataFrame({'query_transcript': [...], 'query_go_terms': [...], 'bart_mfc_acc_diff': [...]})

# Filter the GO terms for aspect (e.g., 'molecular_function') and exclude obsolete terms
molecular_function_terms = {
    term for term, go_obj in go_dag.items()
    if (go_obj.namespace == "molecular_function") and (not getattr(go_obj, 'is_obsolete', False))
}

test_df["filtered_go_terms"] = test_df["query_go_terms"].apply(
    lambda go_terms: [term for term in go_terms if term in molecular_function_terms]
)


# Prepare GSEA input: ranking file and gene set file
# Prepare the ranking file: 'query_transcript' as index and 'bart_mfc_acc_diff' as the metric
ranking = test_df.set_index("query_transcript")["bart_mfc_acc_diff"]

# 2. Gene set file: map GO terms to transcripts
gene_sets = test_df.explode("filtered_go_terms").dropna(subset=["filtered_go_terms"])
gene_sets = gene_sets.groupby("filtered_go_terms")["query_transcript"].apply(list).to_dict()


# Perform GSEA using gseapy
gsea_results = gsea(
    data=ranking,
    gene_sets=gene_sets,
    permutation_num=1000,  # Number of permutations for statistical significance
    outdir="gsea_results",  # Output directory
    format="png",  # Save results as PNG images
    seed=123  # Reproducibility
)

# View results
print("GSEA Results:\n", gsea_results.res2d)


# Define Gene Set Namespaces

In [ ]:
namespaces = ['GO_Biological_Process_2023', 'GO_Cellular_Component_2023', 'GO_Molecular_Function_2023', 'GTEx_Aging_Signatures_2021']
gp.get_library_name()

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down',
 'Allen_Brain_Atlas_up',
 'Azimuth_2023',
 'Azimuth_Cell_Types_2021',
 'BioCarta_2013',
 'BioCarta_2015',
 'BioCarta_2016',
 'BioPlanet_2019',
 'BioPlex_2017',
 'CCLE_Proteomics_2020',
 'CORUM',
 'COVID-19_Related_Gene_Sets',
 'COVID-19_Related_Gene_Sets_2021',
 'Cancer_Cell_Line_Encyclopedia',
 'CellMarker_2024',
 'CellMarker_Augmented_2021',
 'ChEA_2013',
 'ChEA_2015',
 'ChEA_2016',
 'ChEA_2022',
 'Chromosome_Location',
 'Chromosome_Location_hg19',
 'ClinVar_2019',
 'DGIdb_Drug_Targets_2024',
 'DSigDB',
 'Data_Acquisition_Method_Most_Popular_Genes',
 'DepMap_CRISPR_GeneDependency_CellLines_2023',
 'DepMap_WG_CRISPR_Screens_Broad_CellLines_2019',
 'DepMap_WG_CRISPR_Screens_Sanger_Cell